# Import et préparation

In [18]:
# ===============================
# 📌 Section 1 : Import des librairies & Chargement des données
# ===============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Affichage plus lisible
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# Exemple : lecture CSV
data = pd.read_csv("dataset.csv")

# Aperçu
data.head()

# Infos générales

In [ ]:
# ===============================
# 📌 Section 2 : Aperçu global du dataset
# ===============================
print(data.info())
print("\nShape :", data.shape)
print("\nStatistiques descriptives :")
display(data.describe(include="all"))

# Nettoyage de données

In [ ]:
# ===============================
# 📌 Section 3 : Nettoyage de données
# ===============================

# Conversion de date
# data["date_col"] = pd.to_datetime(data["date_col"], errors="coerce")

# Renommer des colonnes
# data.rename(columns={"old_name": "new_name"}, inplace=True)

# Remplir les valeurs manquantes
# data["col"].fillna(data["col"].median(), inplace=True)

# Supprimer les valeurs manquantes
# data.dropna(subset=["col"], inplace=True)

In [ ]:
# ===============================
# 📌 Cellule 3 : Nettoyage complet (doublons, NaN, renommage, imputation simple)
# ===============================

import pandas as pd
import numpy as np
from pandas.api.types import is_numeric_dtype

# 0) Standardisation des noms de colonnes (optionnel mais pratique)
data.columns = [c.strip().lower().replace(" ", "_") for c in data.columns]

# 1) Doublons
print("Lignes avant suppression des doublons :", len(data))
n_dup = data.duplicated().sum()
print("Doublons détectés :", n_dup)
if n_dup > 0:
    data = data.drop_duplicates()
    print("Lignes après suppression des doublons :", len(data))

# 2) Aperçu des valeurs manquantes
missing_count = data.isnull().sum().sort_values(ascending=False)
missing_pct = (data.isnull().mean() * 100).sort_values(ascending=False)
missing_table = pd.concat([missing_count, missing_pct], axis=1)
missing_table.columns = ["missing_count", "missing_percent"]
display(missing_table[missing_table["missing_count"] > 0])

# 3) Suppression des colonnes très vides (ex : >50% NaN)
cols_to_drop = missing_table[missing_table["missing_percent"] > 50].index.tolist()
if cols_to_drop:
    print("Colonnes supprimées (>50% NaN) :", cols_to_drop)
    data = data.drop(columns=cols_to_drop)


# 5) Option : supprimer les lignes avec NaN sur des colonnes critiques (décommenter si besoin)
# critical_cols = ["target", "important_col"]
# data = data.dropna(subset=critical_cols)

print("Shape après nettoyage initial :", data.shape)

# Détection et gestion des valeurs manquantes

In [ ]:
# ===============================
# 📌 Section 4 : Valeurs manquantes
# ===============================
missing_percentages = data.isnull().sum() / len(data) * 100
print(missing_percentages[missing_percentages > 0])

# Analyse des variables numériques

In [ ]:
# ===============================
# 📈 Cellule 6 : Analyse des variables numériques
# ===============================
num_cols = data.select_dtypes(include=[np.number]).columns

# Distribution des variables numériques
for col in num_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(data[col], kde=True, bins=30)
    plt.title(f"Distribution de {col}")
    plt.show()
# Corrélations
print("\nCorrélations :")
display(data[num_cols].corr())

# Heatmap des corrélations
plt.figure(figsize=(10, 8))
sns.heatmap(data[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Heatmap des corrélations")
plt.show()

# Analyse des variables catégorielles

In [ ]:
# ===============================
# 📊 Cellule 7 : Analyse des variables catégorielles
# ===============================
cat_cols = data.select_dtypes(include=["object", "category"]).columns

for col in cat_cols:
    plt.figure(figsize=(8, 4))
    sns.countplot(y=data[col], order=data[col].value_counts().index)
    plt.title(f"Répartition de {col}")
    plt.show()

# Détection des outliers


###  Cellule 8a (IQR avec boucle)

In [ ]:
# Détection outliers avec IQR (toutes colonnes numériques)
num_cols = data.select_dtypes(include=np.number).columns

for col in num_cols:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = data[(data[col] < lower) | (data[col] > upper)]
    print(f"{col}: {len(outliers)} outliers")

###  Cellule 8a-bis (IQR sans boucle, une seule variable)

In [ ]:
# Détection outliers pour une seule variable (sans boucle)
colonne_outlier = num_cols[0]  # changer si besoin
Q1 = data[colonne_outlier].quantile(0.25)
Q3 = data[colonne_outlier].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = data[(data[colonne_outlier] < lower) | (data[colonne_outlier] > upper)]
print(f"{colonne_outlier}: {len(outliers)} outliers")

# Visualisations

### Histogrammes

In [ ]:
for col in num_cols:
    plt.figure(figsize=(6, 4))
    sns.histplot(data[col], kde=True, bins=30)
    plt.title(f"Distribution de {col}")
    plt.show()

In [ ]:
# Recalcule num_cols au besoin
num_cols = data.select_dtypes(include=[np.number]).columns.tolist()
if len(num_cols) == 0:
    raise ValueError("Aucune colonne numérique détectée.")

col_hist = num_cols[0]  # change le nom si tu veux une autre colonne
plt.figure(figsize=(8, 4))
sns.histplot(data[col_hist].dropna(), kde=True, bins=30)
plt.title(f"Distribution de {col_hist}")
plt.xlabel(col_hist)
plt.ylabel("Effectif")
plt.show()

### Boxplots

In [ ]:
for col in num_cols:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=data[col])
    plt.title(f"Boxplot de {col}")
    plt.show()

In [ ]:
# Recalcule num_cols au besoin
num_cols = data.select_dtypes(include=[np.number]).columns.tolist()
if len(num_cols) == 0:
    raise ValueError("Aucune colonne numérique détectée.")

col_box = num_cols[0]  # change le nom si tu veux une autre colonne
plt.figure(figsize=(8, 4))
sns.boxplot(x=data[col_box].dropna())
plt.title(f"Boxplot de {col_box}")
plt.xlabel(col_box)
plt.show()

### Pie chart hors boucle

In [ ]:
colonne_exemple = data.select_dtypes("object").columns[0]
plt.figure(figsize=(6, 6))
data[colonne_exemple].value_counts().plot.pie(autopct="%1.1f%%", startangle=90)
plt.title(f"Répartition de {colonne_exemple}")
plt.ylabel("")
plt.show()

### Scatter plot hors boucle

In [ ]:
col_x, col_y = num_cols[0], num_cols[1]
plt.figure(figsize=(6, 6))
sns.scatterplot(x=data[col_x], y=data[col_y])
plt.title(f"{col_x} vs {col_y}")
plt.show()

# Bootstrapping & Inférences

### Cellule 10a (Bootstrapping simple)

In [ ]:
col = num_cols[0]
values = data[col].dropna()
n_iter = 2000
means = []

for _ in range(n_iter):
    sample = np.random.choice(values, size=len(values), replace=True)
    means.append(np.mean(sample))

plt.figure(figsize=(8, 5))
sns.histplot(means, kde=True, bins=30)
plt.axvline(np.mean(values), color="red", linestyle="--", label="Moyenne réelle")
plt.title(f"Bootstrapping de la moyenne ({col})")
plt.legend()
plt.show()

### Cellule 11 (Test t)

In [ ]:
groupe_col = data.select_dtypes("object").columns[0]
if data[groupe_col].nunique() == 2:
    g1, g2 = data[groupe_col].unique()
    vals1 = data[data[groupe_col] == g1][col].dropna()
    vals2 = data[data[groupe_col] == g2][col].dropna()
    t, p = stats.ttest_ind(vals1, vals2)
    print(f"Test t : {g1} vs {g2} → T={t:.2f}, p={p:.4f}")

### Cellule 12 (Intervalle de confiance)

In [ ]:
mean = np.mean(values)
std = np.std(values, ddof=1)
n = len(values)

ic95 = stats.t.interval(0.95, df=n - 1, loc=mean, scale=std / np.sqrt(n))
print(f"Moyenne = {mean:.2f}, IC95% = [{ic95[0]:.2f}, {ic95[1]:.2f}]")

In [ ]:
# ===============================
# 🧹 Cellule 3 : Nettoyage des données
# ===============================

# 1. Suppression des doublons
print(f"Lignes avant suppression des doublons : {data.shape[0]}")
data = data.drop_duplicates()
print(f"Lignes après suppression des doublons : {data.shape[0]}")

# 2. Gestion des valeurs manquantes
# -> Exemple : suppression des colonnes avec +50% de NaN
missing_percent = data.isnull().mean()
cols_to_drop = missing_percent[missing_percent > 0.5].index
data = data.drop(columns=cols_to_drop)
print(f"Colonnes supprimées (>50% NaN) : {list(cols_to_drop)}")

# -> Exemple : imputation simple (numérique = moyenne, catégoriel = mode)
for col in data.columns:
    if data[col].isnull().sum() > 0:
        if data[col].dtype in [np.float64, np.int64]:
            data[col] = data[col].fillna(data[col].mean())
        else:
            data[col] = data[col].fillna(data[col].mode()[0])

print("✅ Valeurs manquantes traitées")

# 3. Standardisation des noms de colonnes (optionnel)
# data.columns = [col.strip().lower().replace(" ", "_") for col in data.columns]

# 4. Encodage des variables catégorielles (optionnel, pour modélisation)
# data_encoded = pd.get_dummies(data, drop_first=True)

print("✅ Données prêtes après nettoyage")

In [ ]:
# ===============================
# 📊 Cellule 4 : Statistiques générales
# ===============================
print("\nRésumé statistique :")
display(data.describe(include="all"))

In [ ]:
# ===============================
# 🔎 Cellule 5 : Valeurs manquantes (post-nettoyage)
# ===============================
missing_values = data.isnull().sum().sort_values(ascending=False)
missing_percent = (data.isnull().sum() / len(data)) * 100
missing_table = pd.DataFrame(
    {"Missing Values": missing_values, "Percent": missing_percent}
)
display(missing_table)

# Heatmap des valeurs manquantes
# plt.figure(figsize=(10,6))
# sns.heatmap(data.isnull(), cbar=False, cmap="viridis")
# plt.title("Valeurs manquantes")
# plt.show()

In [ ]:
# ===============================
# 📈 Cellule 6 : Analyse des variables numériques
# ===============================
num_cols = data.select_dtypes(include=[np.number]).columns

# Distribution des variables numériques
for col in num_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(data[col], kde=True, bins=30)
    plt.title(f"Distribution de {col}")
    plt.show()

# Corrélations
print("\nCorrélations :")
display(data[num_cols].corr())

# Heatmap des corrélations
plt.figure(figsize=(10, 8))
sns.heatmap(data[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Heatmap des corrélations")
plt.show()

In [ ]:
# ===============================
# 📊 Cellule 7 : Analyse des variables catégorielles
# ===============================
cat_cols = data.select_dtypes(include=["object", "category"]).columns

for col in cat_cols:
    plt.figure(figsize=(8, 4))
    sns.countplot(y=data[col], order=data[col].value_counts().index)
    plt.title(f"Répartition de {col}")
    plt.show()

In [ ]:
# ===============================
# 📊 Cellule 7b : Pie chart pour variables catégorielles
# ===============================

for col in cat_cols:
    plt.figure(figsize=(6, 6))
    data[col].value_counts().plot.pie(
        autopct="%1.1f%%",  # pourcentage
        startangle=90,  # commence à 90°
        counterclock=False,  # ordre horaire
        colormap="Set3",  # palette sympa
    )
    plt.title(f"Répartition de {col}")
    plt.ylabel("")  # enlever l’étiquette "y"
    plt.show()

    # ===============================
# 📊 Cellule 7b : Pie chart (hors boucle)
# ===============================

colonne_exemple = cat_cols[0]  # tu peux changer l'index ou mettre le nom direct

plt.figure(figsize=(6, 6))
data[colonne_exemple].value_counts().plot.pie(
    autopct="%1.1f%%", startangle=90, counterclock=False, colormap="Set3"
)
plt.title(f"Répartition de {colonne_exemple}")
plt.ylabel("")
plt.show()

In [ ]:
# ===============================
# 📊 Cellule 7c : Scatter plots avec boucle
# ===============================

# Ici on compare toutes les colonnes numériques entre elles
for i in range(len(num_cols)):
    for j in range(i + 1, len(num_cols)):  # éviter doublons et x=y
        plt.figure(figsize=(6, 6))
        sns.scatterplot(x=data[num_cols[i]], y=data[num_cols[j]], alpha=0.7)
        plt.title(f"Scatter plot : {num_cols[i]} vs {num_cols[j]}")
        plt.show()


# ===============================
# 📊 Cellule 7C : Scatter plot hors boucle
# ===============================

colonne_x = num_cols[0]  # première variable numérique (modifiable)
colonne_y = num_cols[1]  # deuxième variable numérique (modifiable)

plt.figure(figsize=(6, 6))
sns.scatterplot(x=data[colonne_x], y=data[colonne_y], alpha=0.7)
plt.title(f"Scatter plot : {colonne_x} vs {colonne_y}")
plt.show()

In [ ]:
# ===============================
# ⚠️ Cellule 8 : Détection des valeurs aberrantes
# ===============================
for col in num_cols:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=data[col])
    plt.title(f"Boîte à moustaches - {col}")
    plt.show()

In [ ]:
# ===============================
# ⚠️ Cellule 8a : Détection des valeurs aberrantes (IQR)
# ===============================


def detect_outliers_iqr(df, col):
    """
    Détection des outliers avec la méthode de l'IQR
    Retourne les index des lignes considérées comme outliers
    """
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)].index
    return outliers


outliers_dict = {}

for col in num_cols:
    outliers = detect_outliers_iqr(data, col)
    outliers_dict[col] = len(outliers)
    print(f"{col} : {len(outliers)} outliers détectés")

# Résumé global
outlier_summary = pd.DataFrame.from_dict(
    outliers_dict, orient="index", columns=["Nb_outliers"]
)
display(outlier_summary.sort_values(by="Nb_outliers", ascending=False))

In [ ]:
# ===============================
# ⚠️ Cellule 8a : Détection & Suppression des outliers (IQR)
# ===============================


def remove_outliers_iqr(df, columns):
    """
    Supprime les outliers d'un DataFrame pour les colonnes numériques spécifiées
    en utilisant la méthode de l'IQR.
    Retourne un DataFrame filtré et un résumé des outliers supprimés.
    """
    outlier_summary = {}
    df_clean = df.copy()

    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Comptage des outliers
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        outlier_summary[col] = len(outliers)

        # Filtrage des valeurs
        df_clean = df_clean[
            (df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)
        ]

    return df_clean, pd.DataFrame.from_dict(
        outlier_summary, orient="index", columns=["Nb_outliers"]
    )


# Application sur toutes les colonnes numériques
data_no_outliers, outlier_summary = remove_outliers_iqr(data, num_cols)

print("✅ Outliers supprimés avec la méthode IQR")
print(f"Shape original : {data.shape}")
print(f"Shape nettoyé   : {data_no_outliers.shape}")

# Résumé du nombre d'outliers supprimés par variable
display(outlier_summary.sort_values(by="Nb_outliers", ascending=False))

In [ ]:
# ===============================
# 🔗 Cellule 9 : Analyse bivariée (numérique vs catégoriel)
# ===============================
for cat in cat_cols:
    for num in num_cols:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=data[cat], y=data[num])
        plt.title(f"{num} en fonction de {cat}")
        plt.xticks(rotation=45)
        plt.show()

In [ ]:
# ===============================
# 💾 Cellule 10 : Sauvegarde des résultats (optionnel)
# ===============================
data.describe(include="all").to_csv("eda_summary.csv")
print(
    "✅ Résumé statistique exporté sous 'eda_summary.csv'"
)  # ===============================


In [ ]:
# ===============================
# 📊 Cellule 11 : Bootstrapping
# ===============================

import numpy as np

# Exemple sur une variable numérique
colonne_bootstrap = num_cols[0]  # tu peux changer de colonne
data_col = data[colonne_bootstrap].dropna()


# Fonction bootstrap
def bootstrap(data, n_iterations=1000, sample_size=None):
    if sample_size is None:
        sample_size = len(data)
    means = []
    for _ in range(n_iterations):
        sample = np.random.choice(data, size=sample_size, replace=True)
        means.append(np.mean(sample))
    return np.array(means)


bootstrap_means = bootstrap(data_col, n_iterations=2000)

# Visualisation de la distribution bootstrap
plt.figure(figsize=(8, 5))
sns.histplot(bootstrap_means, kde=True, bins=30)
plt.axvline(np.mean(data_col), color="red", linestyle="--", label="Moyenne réelle")
plt.title(f"Distribution Bootstrap de la moyenne ({colonne_bootstrap})")
plt.legend()
plt.show()


# ===============================
# 📊 Cellule 11a : Bootstrapping (version simple)
# ===============================

# Exemple sur une variable numérique
colonne_bootstrap = num_cols[0]  # tu peux changer de colonne
data_col = data[colonne_bootstrap].dropna()

n_iterations = 2000
sample_size = len(data_col)

# Liste des moyennes bootstrap
bootstrap_means = []

for _ in range(n_iterations):
    sample = np.random.choice(data_col, size=sample_size, replace=True)
    bootstrap_means.append(np.mean(sample))

bootstrap_means = np.array(bootstrap_means)

# Visualisation
plt.figure(figsize=(8, 5))
sns.histplot(bootstrap_means, kde=True, bins=30)
plt.axvline(np.mean(data_col), color="red", linestyle="--", label="Moyenne réelle")
plt.title(f"Distribution Bootstrap de la moyenne ({colonne_bootstrap})")
plt.legend()  # placer la légende
plt.show()

In [ ]:
# ===============================
# 📊 Cellule 11 : Inférences (test t)
# ===============================

from scipy import stats

# Exemple : comparer une variable numérique entre 2 groupes (catégorie binaire)
colonne_test = num_cols[0]
groupe_col = cat_cols[0]

groupes = data[groupe_col].dropna().unique()

if len(groupes) == 2:  # Test t entre 2 groupes seulement
    g1 = data[data[groupe_col] == groupes[0]][colonne_test].dropna()
    g2 = data[data[groupe_col] == groupes[1]][colonne_test].dropna()

    t_stat, p_val = stats.ttest_ind(g1, g2)

    print(f"Test t entre {groupes[0]} et {groupes[1]} sur '{colonne_test}'")
    print(f"T-statistique : {t_stat:.4f}, p-valeur : {p_val:.4f}")

    if p_val < 0.05:
        print("➡️ Différence significative (au seuil de 5%)")
    else:
        print("➡️ Aucune différence significative (au seuil de 5%)")
else:
    print(
        f"La variable '{groupe_col}' n'a pas exactement 2 groupes pour faire un test t."
    )


# Cette fonction génère deux distributions normales, compare leurs histogrammes avec une estimation de densité de noyau, et affiche la p-valeur d'un test t de Student entre elles. Rentrons dans le détail :


def generate_normal_laws(mu_1, mu_2, sample_size):
    # Génère une distribution normale autour de mu_1 avec un écart-type de 2.
    distribution_1 = rng.normal(loc=mu_1, scale=2, size=sample_size)

    # Génère une distribution normale autour de mu_2 avec un écart-type de 2.
    distribution_2 = rng.normal(loc=mu_2, scale=2, size=sample_size)

    # Définit la taille de la figure pour les graphiques.
    sns.set(rc={"figure.figsize": (6, 6)})

    # Crée un histogramme de distribution_1 avec une estimation de densité de noyau.
    ax = sns.histplot(x=distribution_1, kde=True, stat="density", label="samples")

    # Définit le titre du graphique comparant les deux lois normales.
    ax.set_title(
        f"Comparaison des lois normales N({mu_1}, 2) et N({mu_2}, 2), n={sample_size} tirages"
    )

    # Définit l'étiquette de l'axe des x.
    ax.set_xlabel("Panier moyen")

    # Définit l'étiquette de l'axe des y.
    ax.set_ylabel("Proportion dans la population")

    # Ajoute un histogramme de distribution_2 sur le même graphique.
    sns.histplot(x=distribution_2, kde=True, stat="density", label="samples", ax=ax)

    # Affiche la p-valeur d'un test t de Student comparant les deux distributions.
    print(
        f"P-valeur du test : {stats.ttest_ind(distribution_1, distribution_2, random_state=rng).pvalue}"
    )

In [ ]:
# ===============================
# 📊 Cellule 12 : Intervalle de confiance
# ===============================

# Exemple : IC95% de la moyenne d’une variable numérique
colonne_ic = num_cols[0]
data_ic = data[colonne_ic].dropna()

moyenne = np.mean(data_ic)
ecart_type = np.std(data_ic, ddof=1)
n = len(data_ic)

# Intervalle de confiance 95%
ic95 = stats.t.interval(0.95, df=n - 1, loc=moyenne, scale=ecart_type / np.sqrt(n))

print(f"Moyenne de '{colonne_ic}' : {moyenne:.2f}")
print(f"IC95% : [{ic95[0]:.2f}, {ic95[1]:.2f}]")

In [ ]:
# ===============================
# 🛠️ Cellule Bonus : Nettoyage pratique (dates, rename, missing values)
# ===============================

# 1. Conversion en datetime
# Exemple : si tu as une colonne "date" en texte
if "date" in data.columns:
    data["date"] = pd.to_datetime(data["date"], errors="coerce")  # erreurs mises en NaT
    print("✅ Conversion de 'date' en datetime terminée")

# 2. Renommer des colonnes
# Exemple : renommer avec un dictionnaire
rename_dict = {"old_col_name1": "new_col_name1", "old_col_name2": "new_col_name2"}
data = data.rename(columns=rename_dict)
print("✅ Colonnes renommées :", rename_dict)

# 3. Remplir les valeurs manquantes
# Exemple : remplacer par une constante ou une statistique
for col in data.columns:
    if data[col].isnull().sum() > 0:
        if data[col].dtype in [np.float64, np.int64]:
            data[col] = data[col].fillna(data[col].median())  # num = médiane
        else:
            data[col] = data[col].fillna("Inconnu")  # cat = 'Inconnu'
print("✅ Valeurs manquantes remplies")

# 4. Supprimer les lignes avec NaN persistants (optionnel)
# Exemple : si certaines colonnes restent critiques
data = data.dropna()
print("✅ Lignes avec NaN supprimées")

print("Shape final :", data.shape)